In [ ]:
# Séparation des données
X_features = data.drop('Target', axis=1)
y_target = data['Target']

# Échantillonnage 
sample_frac = 0.5
X_sampled = X_features.sample(frac=sample_frac, random_state=42)
y_sampled = y_target.loc[X_sampled.index]

# Pipeline avec PCA et un modèle rapide (LogisticRegression)
pipeline = Pipeline([
    ('pca', PCA()),
    ('clf', GaussianNB())  # Logistic Regression rapide
])

# Grille de paramètres pour PCA (seulement des fractions de composantes)
n_features = X_sampled.shape[1]
param_grid = {
    'pca__n_components': [int(n_features * x) for x in np.linspace(0.1, 1.0, 10)]  # Tester 10 valeurs
}

# Configuration de la validation croisée rapide
cv = StratifiedShuffleSplit(n_splits=5, test_size=0.5, random_state=42)  # 3 splits rapides

# Configuration de la recherche par grille
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    scoring='accuracy',
    cv=cv,
    verbose=1,
    n_jobs=-1  # Utilisation parallèle
)

# Ajustement du modèle avec les données échantillonnées
grid_search.fit(X_sampled, y_sampled)

# Meilleur nombre de composantes PCA
best_n_components = grid_search.best_params_['pca__n_components']
print(f"Meilleur nombre de composantes PCA : {best_n_components}")

# Variance expliquée par les composantes sélectionnées
pca = PCA(n_components=best_n_components)
X_pca_full = pca.fit_transform(X_features)

explained_variance = np.cumsum(pca.explained_variance_ratio_)
print(f"\nVariance cumulée avec {best_n_components} composantes : {explained_variance[-1]:.2%}")

# Visualisation de la variance cumulée
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o')
plt.axhline(y=explained_variance[-1], color='r', linestyle='--', label=f"{explained_variance[-1]:.2%} Variance")
plt.title("Variance Cumulative après PCA")
plt.xlabel("Nombre de Composants")
plt.ylabel("Variance Cumulative")
plt.legend()
plt.grid()
plt.show()



In [ ]:
# Appliquer la PCA avec n_components = 27
n_components = 27
pca = PCA(n_components=best_n_components)
X_selected = pca.fit_transform(X_features)

# Transformation des données
X_pca = pca.fit_transform(X_features)

# Convertir les résultats PCA en DataFrame
X_pca_df = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(n_components)])

# Affichage des 5 premières lignes des données transformées
print("\nLes 5 premières lignes des données après PCA :")
print(X_pca_df.head())



In [ ]:
# Création d'un modèle de classification
clf = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)  # Réduction des estimators pour accélérer

# Validation croisée avec 3 plis
cv_scores = cross_val_score(clf, X_selected, y_target, cv=3, n_jobs=-1)

# Résultats
print("\nScores de validation croisée (3 plis) :", cv_scores)
print("Score moyen de la validation croisée :", np.mean(cv_scores))

In [ ]:
# Prétraitement pour modèles
X_selected = X_pca_df
y_target = data['Target']
X_train, X_test, y_train, y_test = train_test_split(X_selected, y_target, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
preprocessor = ColumnTransformer(
    transformers=[
        ('num', scaler, num_cols),
        ('cat', 'passthrough', df_cat_encoded.columns)
    ]
)